# Week 4: Supervised Learning & Regression
## Notebook 13: Isotonic Regression & Pool Adjacent Violators Algorithm (PAVA)

### 1. Conceptual Intuition & Theoretical Background
Many machine learning models (e.g. Naive Bayes, Boosted Trees) output scores that correctly rank instances but produce **poorly calibrated probabilities** (e.g. a predicted probability of 0.8 only converts 40% of the time).

**Isotonic Regression** fits a free-form, **non-parametric piecewise constant monotonic function**:
$$\min_{\hat{y}} \sum_{i=1}^{n} w_i (y_i - \hat{y}_i)^2 \quad \text{subject to } \hat{y}_1 \le \hat{y}_2 \le \dots \le \hat{y}_n$$

Solved in $O(N)$ linear time via the **Pool Adjacent Violators Algorithm (PAVA)**:
1. Initialize each data point as its own monotonic block.
2. Whenever a monotonicity violation occurs ($\hat{y}_i > \hat{y}_{i+1}$), merge the two adjacent blocks and replace their values with their weighted average.
3. Propagate backward until monotonicity is restored globally.

---

### 2. Scratch Python Implementation: PAVA Engine

In [ ]:
import numpy as np

def pool_adjacent_violators(y_values, weights=None):
    """
    Scratch implementation of Pool Adjacent Violators Algorithm (PAVA).
    Finds monotonic increasing sequence minimizing weighted squared error.
    """
    n = len(y_values)
    if weights is None:
        weights = np.ones(n)
        
    # Block representations: [value, weight, start_idx, end_idx]
    blocks = [[float(y), float(w), i, i] for i, (y, w) in enumerate(zip(y_values, weights))]
    
    i = 0
    while i < len(blocks) - 1:
        # Check for monotonicity violation
        if blocks[i][0] > blocks[i + 1][0]:
            # Pool blocks by calculating weighted average
            val_pooled = (blocks[i][0] * blocks[i][1] + blocks[i+1][0] * blocks[i+1][1]) / (blocks[i][1] + blocks[i+1][1])
            w_pooled = blocks[i][1] + blocks[i+1][1]
            start_idx = blocks[i][2]
            end_idx = blocks[i+1][3]
            
            # Replace current block with pooled block and delete next block
            blocks[i] = [val_pooled, w_pooled, start_idx, end_idx]
            blocks.pop(i + 1)
            
            # Check backwards to resolve any newly created violations
            while i > 0 and blocks[i - 1][0] > blocks[i][0]:
                val_back = (blocks[i-1][0] * blocks[i-1][1] + blocks[i][0] * blocks[i][1]) / (blocks[i-1][1] + blocks[i][1])
                w_back = blocks[i-1][1] + blocks[i][1]
                start_back = blocks[i-1][2]
                end_back = blocks[i][3]
                
                blocks[i - 1] = [val_back, w_back, start_back, end_back]
                blocks.pop(i)
                i -= 1
        else:
            i += 1
            
    # Unpack fitted step function
    y_iso = np.zeros(n)
    for b in blocks:
        y_iso[b[2] : b[3] + 1] = b[0]
        
    return y_iso

# Demonstration: Calibrating Uncalibrated Raw Model Prediction Scores
np.random.seed(42)
n_samples = 40
raw_scores = np.sort(np.random.uniform(0.1, 0.9, n_samples))
# Noisy non-monotonic binary conversions
binary_labels = np.random.binomial(1, raw_scores ** 2)

calibrated_probs = pool_adjacent_violators(binary_labels)

# Verify strict monotonicity
is_monotonic = np.all(np.diff(calibrated_probs) >= 0)

print("--- Isotonic Regression Calibration (PAVA) ---")
print(f"Sample Binary Targets:      {binary_labels[:10]}")
print(f"Calibrated Probabilities:   {np.round(calibrated_probs[:10], 3)}")
print(f"Monotonicity Verified:      {is_monotonic}")
print(f"Unique Probability Buckets: {len(np.unique(calibrated_probs))}")

### 3. Step-by-Step Code Walkthrough
1. **Violator Detection:** Scans for local inversions where a lower-scoring instance has a higher value.
2. **Backward Pooling:** Merges violating blocks into weighted averages, propagating backward until monotonicity holds.
3. **Calibrated Mapping:** Transforms uncalibrated raw scores into verified monotonic posterior probabilities.

---

### 4. Industry Applications & Trade-offs
- **Real-Time Bidding (RTB) Ad Auctions:** Calibrating predicted CTRs for downstream expected value bidding.
- **Trade-off:** Prone to overfitting on small datasets ($N < 1,000$); Platt Scaling is preferred when data is sparse.

---

### 5. Top Interview Questions & Answers
**Q1: When should you use Isotonic Regression over Platt Scaling for probability calibration?**
> **Answer:** Use Isotonic Regression when you have sufficient validation data ($N > 1,000$) and the calibration mapping is non-sigmoid. Use Platt Scaling if sample sizes are small to avoid overfitting.

**Q2: What is the computational time complexity of the PAVA algorithm?**
> **Answer:** Assuming the data is already sorted by predicted score, PAVA runs in $O(N)$ linear time because each block is created and merged at most once.